In [1]:
import numpy as np
from random import random, seed

# Funkcje grafowe

In [2]:
def print_matrix(vertices, matrix):
  """
  Wypisuje na ekranie graf podany jako macierz sąsiedztwa
  """
  n = len(matrix)
  if (vertices is None) or (len(vertices) != n):
    vv = range(1, n+1)
  else:
    vv = vertices
  for i in range(n):
    print(vv[i], ':', end='')
    for j in range(n):
      if (matrix[i][j]):
        print(" ", vv[j], end="")
    print("")

In [3]:
def print_graph(graph):
  """
  Wypisuje na ekranie graf podany jako słownik (list) sąsiedztwa
  """
  for v in graph:
    print(v, ':', end="")
    for u in graph[v]:
      print(" ", u, end="")
    print("")


## Modyfikacje grafów

In [4]:
def add_vertex(graph, vertex):
  """
  Dodaje wierzchołek do grafu
  """
  if vertex not in graph:
    graph[vertex] = []

def add_arc(graph, arc):
  """
  Dodaje łuk (skierowany, podany jako para wierzchołków) do grafu
  """
  u, v = arc
  add_vertex(graph, u)
  add_vertex(graph, v)
  if v not in graph[u]:
    graph[u].append(v)

def add_edge(graph, edge):
  """
  Dodaje krawędź (podaną jako para wierzchołków) do grafu
  Rozpatrujemy grafy proste, nieskierowane
  """
  u, v = edge
  add_vertex(graph, u)
  add_vertex(graph, v)
  if u == v:
    raise ValueError("Pętla!!!")
  if v not in graph[u]:
    graph[u].append(v)
  if u not in graph[v]:
    graph[v].append(u)



## Zapis i odczyt z plików

In [5]:
def graph_from_edges(filename, directed=0):
  """
  Tworzy graf na podstawie pliku z krawędziami. Opis krawędzi to dwa słowa lub wierzchołka (jedno słowo).
  Nadmiarowe słowa są ignorowane.
  Zmienna filename zawiera pełną ścieżkę pliku
  """
  graph = {}
  with open(filename, 'r') as file: # otwarcie do odczytu
    for line in file:
      words = line.strip().split()
      if len(words) == 1:     # jedno słowo - opis wierzchołka
        add_vertex(graph, words[0])
      elif len(words) >= 2:   # co najmniej dwa słowa - opis krawędzi
        if directed:
          add_arc(graph, (words[0], words[1]))
        else:
          add_edge(graph, (words[0], words[1]))
  return graph

In [6]:
def graph_to_neighbourlist(graph, filename):
  """
  Zapisuje graf podany jako słownik (list) sąsiedztwa do pliku (w formie listy sąsiedztwa).
  Zmienna filename zawiera pełną ścieżkę pliku
  """
  with open(filename, 'w') as file: # otwarcie do zapisu
    for v in graph:
      line = f"{v}:"
      for u in graph[v]:
        line += f" {u}"
      line += "\n"
      file.write(line)

In [17]:
# zapisywanie grafu do pliku w postaci listy krawędzi,
def graph_to_edges(graph, filename, directed=0):
  with open(filename, 'w') as file:
    visited = set() # żeby się nie powtarzały krawędzie
    for v in graph:
      for u in graph[v]:
        if directed:
          file.write(f"{v} {u}\n") # każda krawędź w osobnym wierszu
        else:
          if (u, v) not in visited:
            file.write(f"{v} {u}\n")
            visited.add((v, u))

In [19]:
# wczytywanie grafu z listy sąsiedztwa
def graph_from_neighbourlist(filename, directed=0):
  graph = {}

  with open(filename, 'r') as file:
    for line in file:
      parts = line.strip().split(":")

      v = parts[0].strip()
      add_vertex(graph, v)

      if len(parts) > 1:
        neighbours = parts[1].strip().split()

        for u in neighbours:
          if directed:
            add_arc(graph, (v,u))
          else:
            add_edge(graph, (v,u))
  return graph

# Testowanie funkcji

## Wczytywanie i zapis grafów

In [7]:
# zapisuje to co jest poniżej do pliku
%%writefile edges.txt
a b
a c
b d
c e
f

Writing edges.txt


In [8]:
%cat edges.txt

a b
a c
b d
c e
f


In [21]:
graph1 = graph_from_edges('edges.txt')

In [22]:
print_graph(graph1)

a :  b  c
b :  a  d
c :  a  e
d :  b
e :  c
f :


In [23]:
digraph1 = graph_from_edges('edges.txt', directed=1)
print_graph(digraph1)

a :  b  c
b :  d
c :  e
d :
e :
f :


In [24]:
!wget https://raw.githubusercontent.com/pgordin/OptDisc2026/refs/heads/main/weighted0.txt

--2026-04-19 19:02:27--  https://raw.githubusercontent.com/pgordin/OptDisc2026/refs/heads/main/weighted0.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 114 [text/plain]
Saving to: ‘weighted0.txt.1’

weighted0.txt.1     100%[===================>]     114  --.-KB/s    in 0s      

2026-04-19 19:02:27 (6.40 MB/s) - ‘weighted0.txt.1’ saved [114/114]



In [25]:
%cat weighted0.txt

A B 3
A E 10
B C 26
B D 12
C D 17
C F 13
C G 14
D E 7
D F 15
E F 8
E H 4
F G 9
F H 6
G H 16
G I 11


In [26]:
graph2 = graph_from_edges('weighted0.txt')
print_graph(graph2)

A :  B  E
B :  A  C  D
E :  A  D  F  H
C :  B  D  F  G
D :  B  C  E  F
F :  C  D  E  G  H
G :  C  F  H  I
H :  E  F  G
I :  G


In [33]:
graph_to_neighbourlist(graph2, 'graph2.txt')

In [28]:
%cat graph2.txt

A: B E
B: A C D
E: A D F H
C: B D F G
D: B C E F
F: C D E G H
G: C F H I
H: E F G
I: G


In [29]:
graph_to_edges(graph1, "edges2.txt")

In [31]:
%cat edges2.txt

a b
a c
b d
c e


In [35]:
graph_dict = graph_from_neighbourlist("graph2.txt")
print_graph(graph_dict)

A :  B  E
B :  A  C  D
E :  A  D  F  H
C :  B  D  F  G
D :  B  E  C  F
F :  E  C  D  G  H
H :  E  F  G
G :  C  F  H  I
I :  G
